In [18]:
import numpy as np
import time
import platform
from pathlib import Path
from skimage import io
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf
from PIL import Image
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, AveragePooling2D, Dense, Flatten, InputLayer
from tqdm import tqdm
import os
import gc
from sklearn.preprocessing import LabelEncoder


In [2]:
# if torch.cuda.is_available():
#     device = 'cuda'
# elif torch.mps.is_available():
#     device = 'mps'
# else:
#     'cpu'
    
# print(device)

devices = tf.config.list_physical_devices()
print("\nDevices: ", devices)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
  details = tf.config.experimental.get_device_details(gpus[0])
  print("GPU details: ", details)


Devices:  [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU details:  {'device_name': 'METAL'}


In [3]:
import kagglehub

path = kagglehub.competition_download('2-computer-vision-2026-b-sc-aidams-final-proj')

print("Path to competition files:", path)

/Users/iamgeorgerieh/Documents/computer-vision-class/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 248M/248M [00:06<00:00, 39.4MB/s] 

Extracting files...


Path to competition files: /Users/iamgeorgerieh/.cache/kagglehub/competitions/2-computer-vision-2026-b-sc-aidams-final-proj


In [ ]:
# def load_data(folder_path:str):
#     path_to_images = Path(folder_path) 
#     ids = []
#     dataset = []
#     for img_file in sorted(path_to_images.iterdir()):
#         id_img = "".join(img_file.name.split(".")[:-1])
#         ids.append(id_img)
#         im = Image.open(img_file)
#         # if im.format == 'PNG' and im.mode != 'RGBA': 
#             # encountered multiple errors look like UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
#         img_file = f'{img_file}2.png'
#         im = im.convert("RGBA").save(img_file)
        
#         img = io.imread(img_file, as_gray=True)
#         dataset.append(img)
#     dataset_as_array = np.array(dataset, dtype="object")
#     return dataset_as_array, ids

In [4]:
def load_data(folder_path):
    pbar = tqdm(total=len([name for name in os.listdir(folder_path) if os.path.isfile(name)]))
    path_to_images = Path(folder_path)
    ids = []
    dataset = []
    for img_file in sorted(path_to_images.iterdir()):
        id_img = "".join(img_file.name.split(".")[:-1])
        ids.append(id_img)
        im = Image.open(img_file)
        if im.format == 'PNG' and im.mode != 'RGBA':
            img_file = f'{img_file}2.png'
            im.convert("RGBA").save(img_file)
        img = io.imread(img_file)
        gray_weighted = img[:,:,0] * 0.299 + 0.587 * img[:,:,1] + 0.114 * img[:,:,2]
        gray_weighted = tf.image.resize([gray_weighted], [72, 72])
        gray_weighted = tf.image.random_brightness(gray_weighted, max_delta=0.1) 
        gray_weighted = tf.image.random_contrast(gray_weighted, lower=0.9, upper=1.1)
        dataset.append(gray_weighted)
        pbar.update(1)
    
    dataset_as_array = np.array(dataset, dtype='object')
    pbar.close()
    return dataset_as_array, ids


In [4]:
def load_data(folder_path):
    path_to_images = Path(folder_path)
    image_files = [f for f in sorted(path_to_images.iterdir()) if f.is_file()]
    
    ids = []
    dataset = []
    
    pbar = tqdm(total=len(image_files))
    
    for img_file in image_files:
        id_img = img_file.stem
        ids.append(id_img)
        
        with Image.open(img_file) as im:
            if im.format == 'PNG' and im.mode != 'RGBA':
                converted_path = img_file.parent / f"{img_file.stem}_conv.png"
                im.convert("RGBA").save(converted_path)
                img_file = converted_path

        img = io.imread(img_file)
        
        gray_weighted = img[:, :, 0] * 0.299 + 0.587 * img[:, :, 1] + 0.114 * img[:, :, 2]
        
        gray_weighted = np.expand_dims(gray_weighted, axis=-1)
        
        resized_tensor = tf.image.resize(gray_weighted, [72, 72])
        resized_arr = resized_tensor.numpy().astype(np.float32) / 255.0
        
        dataset.append(resized_arr)
        
        del img, gray_weighted, resized_tensor
        pbar.update(1)
    
    pbar.close()
    

    dataset_as_array = np.array(dataset, dtype=np.float32)
    
    gc.collect() #asked ai for this part, because I was doing transformation inside the loop and without garbage collectio n
    
    return dataset_as_array, ids

In [5]:
X_train, X_train_ids = load_data(path+"/train")
print("Shape of train dataset", X_train.shape)

X_test, X_test_ids = load_data(path+"/test")
print("Shape of test dataset", X_test.shape)
y_train = pd.read_csv(path+"/train_labels.csv")
print("Shape of predictions for train dataset", y_train.shape)

  0%|          | 0/9879 [00:00<?, ?it/s]2026-09-17 20:55:42.508240: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1
2026-09-17 20:55:42.508334: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-09-17 20:55:42.508354: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2026-09-17 20:55:42.509170: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-09-17 20:55:42.511318: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
100%|██████████| 9879/9879 [00:21<00:00, 469.25it/s]


Shape of train dataset (9879, 72, 72, 1)


100%|██████████| 9879/9879 [00:20<00:00, 474.93it/s]


Shape of test dataset (9879, 72, 72, 1)
Shape of predictions for train dataset (9879, 2)


In [26]:
# X_train_norm = X_train.astype("float32") / 255.0
# X_test_norm = X_test.astype("float32") / 255.0

# X_train_norm = np.expand_dims(X_train, axis=-1)
# X_test_norm = np.expand_dims(X_test, axis=-1)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (9879, 72, 72, 1)
Test: (9879, 72, 72, 1)


In [78]:
tf.keras.backend.clear_session()

In [79]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomBrightness(factor=0.1),  
    tf.keras.layers.RandomContrast(factor=0.1),    
])

model = Sequential([
    InputLayer(input_shape=(72, 72, 1)),
    # data_augmentation, 
    Conv2D(
        filters= 6,
        kernel_size=(5,5),
        activation='relu',
        strides=(1,1),
        padding='same',),
    
    AveragePooling2D(
        pool_size=(2,2),
        strides=2
    ),
    
    Conv2D(
        filters=16,
        kernel_size=(5,5),
        strides=(1,1),
        padding='valid',
        activation='relu'
    ),
    
    AveragePooling2D(
        pool_size=(2,2),
        strides=2
    ),

    
    Flatten(),

    Dense(120, activation='relu'),
    Dense(84, activation='relu'),
    Dense(7, activation='softmax')
])

model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 72, 72, 6)         156       
                                                                 
 average_pooling2d (Average  (None, 36, 36, 6)         0         
 Pooling2D)                                                      
                                                                 
 conv2d_1 (Conv2D)           (None, 32, 32, 16)        2416      
                                                                 
 average_pooling2d_1 (Avera  (None, 16, 16, 16)        0         
 gePooling2D)                                                    
                                                                 
 flatten (Flatten)           (None, 4096)              0         
                                                                 
 dense (Dense)               (None, 120)              

In [80]:
model.compile(
    loss="sparse_categorical_crossentropy",
    #AI
    # Use sparse_categorical_crossentropy if y_train looks like integers:
    # [0, 3, 1, 9, ...] (Shape: (batch_size, 1) or (batch_size,))
    # Use categorical_crossentropy ONLY if you converted y_train into one-hot binary vectors first:
    # [[1, 0, 0, ...], [0, 0, 1, ...]] (Shape: (batch_size, num_classes))
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00002, clipnorm=1.0),
    metrics=["accuracy"]
)

In [60]:
labels_df = pd.read_csv(path+'/train_labels.csv', dtype={'Id': str})

label_map = dict(zip(labels_df['Id'].astype(str), labels_df['Label']))

y_strings = [label_map[img_id] for img_id in X_train_ids]

In [61]:
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_strings)

class_names = list(label_encoder.classes_)
print("Classes:", class_names)

Classes: ['apple', 'facebook', 'google', 'messenger', 'mozilla', 'samsung', 'whatsapp']


In [62]:
y_train

array([5, 0, 1, ..., 0, 2, 5])

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=64,
    validation_split=0.2,
)

Epoch 1/100


2026-09-17 21:43:31.326041: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node Adam/AssignAddVariableOp.


 36/124 [=======>......................] - ETA: 1s - loss: 1.9890 - accuracy: 0.1393

In [45]:
X_tiny = X_train[:32]
y_tiny = y_train[:32]

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# If the setup works, accuracy MUST hit 100% (1.0) within 30-50 epochs
model.fit(X_tiny, y_tiny, epochs=50, batch_size=32)

Epoch 1/50


2026-09-17 21:30:14.329965: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node Adam/AssignAddVariableOp.
2026-09-17 21:30:14.371836: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation.


1/1 [==============================] - 1s 733ms/step - loss: 1.9588 - accuracy: 0.2188
Epoch 2/50
1/1 [==============================] - 0s 60ms/step - loss: 1.8949 - accuracy: 0.3438
Epoch 3/50
1/1 [==============================] - 0s 20ms/step - loss: 1.8807 - accuracy: 0.2500
Epoch 4/50
1/1 [==============================] - 0s 22ms/step - loss: 1.8169 - accuracy: 0.3438
Epoch 5/50
1/1 [==============================] - 0s 21ms/step - loss: 1.7965 - accuracy: 0.3438
Epoch 6/50
1/1 [==============================] - 0s 23ms/step - loss: 1.8433 - accuracy: 0.3438
Epoch 7/50
1/1 [==============================] - 0s 23ms/step - loss: 1.8513 - accuracy: 0.3438
Epoch 8/50
1/1 [==============================] - 0s 22ms/step - loss: 1.7978 - accuracy: 0.3438


2026-09-17 21:30:14.766941: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation.
2026-09-17 21:30:14.828218: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation.
2026-09-17 21:30:14.851556: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation.
2026-09-17 21:30:14.874837: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation.
2026-09-17 21:30:14.897986: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation.
2026-09-17 21:30:14.923961: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation

Epoch 9/50
1/1 [==============================] - 0s 22ms/step - loss: 1.7541 - accuracy: 0.3438
Epoch 10/50
1/1 [==============================] - 0s 23ms/step - loss: 1.7268 - accuracy: 0.3438
Epoch 11/50
1/1 [==============================] - 0s 20ms/step - loss: 1.7434 - accuracy: 0.3438
Epoch 12/50
1/1 [==============================] - 0s 18ms/step - loss: 1.7925 - accuracy: 0.3438
Epoch 13/50
1/1 [==============================] - 0s 18ms/step - loss: 1.7134 - accuracy: 0.3125
Epoch 14/50
1/1 [==============================] - 0s 26ms/step - loss: 1.7928 - accuracy: 0.3125
Epoch 15/50
1/1 [==============================] - 0s 21ms/step - loss: 1.7551 - accuracy: 0.4062
Epoch 16/50
1/1 [==============================] - 0s 22ms/step - loss: 1.7772 - accuracy: 0.2812
Epoch 17/50


...

2026-09-17 21:30:15.420289: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation.
2026-09-17 21:30:15.442608: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation.
2026-09-17 21:30:15.465861: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation.
2026-09-17 21:30:15.487284: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation.
2026-09-17 21:30:15.506206: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation.
2026-09-17 21:30:15.525409: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation

1/1 [==============================] - 0s 20ms/step - loss: 1.7210 - accuracy: 0.3438
Epoch 38/50
1/1 [==============================] - 0s 24ms/step - loss: 1.7417 - accuracy: 0.3438
Epoch 39/50
1/1 [==============================] - 0s 22ms/step - loss: 1.7200 - accuracy: 0.3438
Epoch 40/50
1/1 [==============================] - 0s 20ms/step - loss: 1.7573 - accuracy: 0.3438
Epoch 41/50
1/1 [==============================] - 0s 19ms/step - loss: 1.7671 - accuracy: 0.3438
Epoch 42/50
1/1 [==============================] - 0s 17ms/step - loss: 1.7366 - accuracy: 0.3438
Epoch 43/50
1/1 [==============================] - 0s 18ms/step - loss: 1.7432 - accuracy: 0.3438
Epoch 44/50
1/1 [==============================] - 0s 17ms/step - loss: 1.7403 - accuracy: 0.3438
Epoch 45/50
1/1 [==============================] - 0s 18ms/step - loss: 1.7374 - accuracy: 0.3438
Epoch 46/50


2026-09-17 21:30:15.621617: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation.
2026-09-17 21:30:15.643369: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation.
2026-09-17 21:30:15.669053: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation.
2026-09-17 21:30:15.691760: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation.
2026-09-17 21:30:15.713140: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation.
2026-09-17 21:30:15.734804: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation

1/1 [==============================] - 0s 21ms/step - loss: 1.6891 - accuracy: 0.3438
Epoch 47/50
1/1 [==============================] - 0s 25ms/step - loss: 1.7189 - accuracy: 0.3438
Epoch 48/50
1/1 [==============================] - 0s 21ms/step - loss: 1.7130 - accuracy: 0.3438
Epoch 49/50
1/1 [==============================] - 0s 22ms/step - loss: 1.7530 - accuracy: 0.3438
Epoch 50/50
1/1 [==============================] - 0s 21ms/step - loss: 1.7404 - accuracy: 0.3438


2026-09-17 21:30:15.834960: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation.
2026-09-17 21:30:15.860527: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation.
2026-09-17 21:30:15.884284: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation.
2026-09-17 21:30:15.907309: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation.


In [39]:
print(X_train.max())

1.0


In [42]:
# 1. Check label range and unique values
import numpy as np
print("Labels shape:", y_train.shape)
print("Unique label values:", np.unique(y_train))  # MUST be array([0, 1, 2, 3, 4, 5, 6])

# 2. Check image array dimensions
print("Images shape:", X_train.shape)  # Last number should be channels (e.g. 1 or 3)

# 3. Check trainable parameters
print("Trainable params:", sum([np.prod(v.get_shape().as_list()) for v in model.trainable_weights]))

Labels shape: (9879,)
Unique label values: [0 1 2 3 4 5 6]
Images shape: (9879, 72, 72, 1)
Trainable params: 504971
